<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
# =============================================================================
# CELL 1: CONNECTION HEALTH GATE
# =============================================================================

import pyodbc

DSN = "Redshift_prod_new"

try:
    with pyodbc.connect(f"DSN={DSN}", timeout=15) as conn:
        conn.execute("SELECT 1")
    print(f"Redshift connection OK  (DSN={DSN})")
except Exception as e:
    raise RuntimeError(f"Cannot connect to Redshift (DSN={DSN}): {e}")

RuntimeError: Cannot connect to Redshift (DSN=Redshift_prod_new): ('HY000', '[HY000] [Redshift][ODBC Driver][Server][860:8:IAMConnectionError]: Authentication failed on the Azure server. Please check the IdP Tenant, User, Password, Client Secret and Client ID. Response code: 400\n (0) (SQLDriverConnect); [HY000] [Redshift][ODBC Driver][Server][860:8:IAMConnectionError]: Authentication failed on the Azure server. Please check the IdP Tenant, User, Password, Client Secret and Client ID. Response code: 400\n (0)')

In [ ]:
# =============================================================================
# CELL 2: IMPORTS AND TABLE CONFIG
# =============================================================================

import concurrent.futures
import pyodbc
import pandas as pd
import time
import os
from IPython.display import display, Markdown

update_tables = True


SAMPLE_ROWS = 5
QUERY_TIMEOUT_SEC = 300

# ---------------------------------------------------------------------------
# Freshness query templates for dateless tables.
# {table} is replaced at runtime with the fully qualified table name.
# ---------------------------------------------------------------------------

LOAN_ID_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.loan_id = cd.loan_id
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

ACCT_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.account_number = cd.account_number
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

CUST_ID_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.customer_id = cd.pb_customer_id
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

CUSTOMERID_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.customerid = cd.pb_customer_id
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

DEALER_NUM_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.dealer_number = cd.dealer_number
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

DEALER_ID_FRESHNESS = """
    SELECT MAX(cd.application_received_date) AS max_dt
    FROM {table} t
    LEFT JOIN edwnpi.los_deal_current_fact cd ON t.dealerid = cd.dealer_number
    WHERE cd.application_received_date >= DATEADD(day, -90, CURRENT_DATE)
"""

# ---------------------------------------------------------------------------
# Table registry -- volatile sandbox tables first, stable edwnpi tables last.
# ---------------------------------------------------------------------------

TABLES_TO_CHECK = [
    # --- Volatile sandbox tables first ---
    {"table": "sandbox.student_loan_chime_flags",                    "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.temp_employment_type_ragu",                   "key_col": "account_number","date_col": None,       "used_by": "ULA",                          "freshness_query": ACCT_FRESHNESS},
    {"table": "sandbox.rds_blackbook_rollup",                        "key_col": "account_number","date_col": None,       "used_by": "ULA",                          "freshness_query": ACCT_FRESHNESS},
    {"table": "sandbox.kmx_approvals",                               "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.kmx_los_new_sp",                              "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.temp_prov_customer_credit_attributes_ragu",   "key_col": "customerid",    "date_col": None,       "used_by": "ULA",                          "freshness_query": CUSTOMERID_FRESHNESS},
    {"table": "sandbox.temp_los_customer_credit_attributes_ragu",    "key_col": "customer_id",   "date_col": None,       "used_by": "ULA",                          "freshness_query": CUST_ID_FRESHNESS},
    {"table": "sandbox.loan_random_numbers",                         "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.temp_fraud_ragu",                             "key_col": "loan_id",       "date_col": None,       "used_by": "ULA",                          "freshness_query": LOAN_ID_FRESHNESS},
    {"table": "sandbox.temp_blackbook_values_ragu",                   "key_col": "account_number","date_col": None,       "used_by": "ULA / Recovery",               "freshness_query": ACCT_FRESHNESS},
    {"table": "sandbox.nonkmx_dealer_loss_data",                     "key_col": "dealer_number", "date_col": None,       "used_by": "DLA",                          "freshness_query": DEALER_NUM_FRESHNESS},
    {"table": "sandbox.rds_rec_model_originations",                  "key_col": "account_number","date_col": "con_date", "used_by": "Recovery",                     "freshness_query": None},

    # --- Stable edwnpi tables last ---
    {"table": "edwnpi.los_deal_current_fact",                        "key_col": "account_number","date_col": "book_date","used_by": "Model Scores / ULA",           "freshness_query": None},
    {"table": "edwnpi.dealer_rollup_scd_current",                    "key_col": "dealer_number", "date_col": None,       "used_by": "Model Scores / ULA",           "freshness_query": DEALER_NUM_FRESHNESS},
    {"table": "edwnpi.date_dim",                                     "key_col": "calendar_date", "date_col": "calendar_date", "used_by": "Model Scores / ULA / Recovery","freshness_query": None},
    {"table": "edwnpi.dealer_attributes_pivot",                      "key_col": "dealerid",      "date_col": None,       "used_by": "ULA",                          "freshness_query": DEALER_ID_FRESHNESS},
    {"table": "edwnpi.crm_dealer_dim",                               "key_col": "dealer_number", "date_col": None,       "used_by": "ULA",                          "freshness_query": DEALER_NUM_FRESHNESS},
]

print(f"Tables to check: {len(TABLES_TO_CHECK)}")
print(f"Sample rows per table: {SAMPLE_ROWS}")
print(f"Query timeout: {QUERY_TIMEOUT_SEC}s")

In [ ]:
# Parameters
update_tables = False


In [ ]:
# =============================================================================
# CELL 3: PARALLEL DISCOVERY PROBE
# =============================================================================

import warnings

def check_table(cfg):
    """Two-pass probe for a single table. Runs in its own thread with its own connection."""
    table = cfg["table"]
    key_col = cfg["key_col"]
    date_col = cfg.get("date_col")
    freshness_query = cfg.get("freshness_query")

    result = {
        "table": table,
        "used_by": cfg["used_by"],
        "reachable": False,
        "total_rows": None,
        "non_null_key_rows": None,
        "sample_df": None,
        "columns": None,
        "max_date": None,
        "recent_rows": None,
        "elapsed_sec": None,
        "error": None,
        "freshness_error": None,
    }

    t0 = time.time()
    try:
        conn = pyodbc.connect(f"DSN={DSN}", timeout=15)
        conn.timeout = QUERY_TIMEOUT_SEC
    except Exception as e:
        result["error"] = f"Connection failed: {str(e)[:300]}"
        result["elapsed_sec"] = round(time.time() - t0, 2)
        return result

    try:
        warnings.filterwarnings("ignore", category=UserWarning)

        # ---- PASS 1: Existence, health, and sample (no joins) ----
        count_q = f"SELECT COUNT(*) AS total_rows, COUNT({key_col}) AS non_null_key_rows FROM {table}"
        count_row = pd.read_sql_query(count_q, conn)
        result["reachable"] = True
        result["total_rows"] = int(count_row["total_rows"].iloc[0])
        result["non_null_key_rows"] = int(count_row["non_null_key_rows"].iloc[0])

        sample_q = f"SELECT * FROM {table} LIMIT {SAMPLE_ROWS}"
        sample_df = pd.read_sql_query(sample_q, conn)
        result["sample_df"] = sample_df
        result["columns"] = list(sample_df.columns)

        # ---- PASS 2: Freshness + recent volume (only if Pass 1 succeeded) ----
        try:
            if date_col:
                fresh_q = (
                    f"SELECT COUNT(*) AS recent_rows, MAX({date_col}) AS max_dt FROM {table} "
                    f"WHERE {date_col} >= DATEADD(day, -90, CURRENT_DATE)"
                )
                fresh_row = pd.read_sql_query(fresh_q, conn)
                result["max_date"] = str(fresh_row["max_dt"].iloc[0])
                result["recent_rows"] = int(fresh_row["recent_rows"].iloc[0])
            elif freshness_query:
                fresh_tmpl = freshness_query.replace("MAX(cd.application_received_date) AS max_dt",
                                                     "COUNT(*) AS recent_rows, MAX(cd.application_received_date) AS max_dt")
                fresh_q = fresh_tmpl.format(table=table)
                fresh_row = pd.read_sql_query(fresh_q, conn)
                result["max_date"] = str(fresh_row["max_dt"].iloc[0])
                result["recent_rows"] = int(fresh_row["recent_rows"].iloc[0])
        except Exception as e:
            result["freshness_error"] = f"Freshness check failed (join dependency may be down): {str(e)[:300]}"

        warnings.filterwarnings("default", category=UserWarning)

    except Exception as e:
        result["error"] = str(e)[:300]
    finally:
        conn.close()
        result["elapsed_sec"] = round(time.time() - t0, 2)

    return result


# ---- Dispatch all table checks in parallel ----
print(f"Probing {len(TABLES_TO_CHECK)} tables in parallel ...\n")
probe_start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(check_table, cfg): cfg["table"] for cfg in TABLES_TO_CHECK}
    probe_results = []
    for future in concurrent.futures.as_completed(futures):
        r = future.result()
        tag = "OK" if r["reachable"] else "FAIL"
        print(f"  [{tag}] {r['table']}  ({r['elapsed_sec']}s)")
        probe_results.append(r)

probe_elapsed = round(time.time() - probe_start, 2)
print(f"\nAll probes complete in {probe_elapsed}s")
print("[PROGRESS] Probe Complete")

In [ ]:
# =============================================================================
# CELL 4: SAMPLE DATA VIEWER
# =============================================================================

pd.set_option("display.max_columns", 50)

for r in sorted(probe_results, key=lambda x: x["table"]):
    table = r["table"]
    sample = r["sample_df"]

    display(Markdown(f"---\n### `{table}`"))

    if r["error"]:
        display(Markdown(f"**Error:** {r['error']}"))
        continue

    info_parts = [
        f"**Used by:** {r['used_by']}",
        f"**Rows:** {r['total_rows']:,}",
        f"**Non-null key rows:** {r['non_null_key_rows']:,}",
        f"**Columns ({len(r['columns'])}):** `{'`, `'.join(r['columns'])}`",
        f"**Max date:** {r['max_date']}",
        f"**Elapsed:** {r['elapsed_sec']}s",
    ]
    if r["freshness_error"]:
        info_parts.append(f"**Freshness error:** {r['freshness_error']}")

    display(Markdown("  \n".join(info_parts)))

    if sample is not None and len(sample) > 0:
        display(sample)
    else:
        display(Markdown("*No sample rows returned.*"))

In [ ]:
# =============================================================================
# CELL 5: QUERY FILE EXISTENCE CHECK
# =============================================================================

QUERY_FILES = [
    ("postmodern_ms_query.txt",    "Model Scores"),
    ("vintage_level_ula_query.txt", "ULA"),
    ("new_dll_query.txt",           "DLA"),
    ("new_recovery_queryt.txt",     "New Recovery"),
]

print("Query file check:")
for filename, label in QUERY_FILES:
    exists = os.path.exists(filename)
    full_path = os.path.abspath(filename)
    status = "EXISTS" if exists else "MISSING"
    print(f"  [{status}]  {filename}  ({label})")
    if not exists:
        print(f"           Expected at: {full_path}")

In [ ]:
# =============================================================================
# CELL 6: SUMMARY DATAFRAME
# =============================================================================

summary_rows = []
for r in probe_results:
    summary_rows.append({
        "table": r["table"],
        "used_by": r["used_by"],
        "reachable": r["reachable"],
        "total_rows": r["total_rows"],
        "non_null_key_rows": r["non_null_key_rows"],
        "num_columns": len(r["columns"]) if r["columns"] else None,
        "max_date": r["max_date"],
        "recent_rows": r["recent_rows"],
        "elapsed_sec": r["elapsed_sec"],
        "error": r["error"],
        "freshness_error": r["freshness_error"],
    })

diag_df = pd.DataFrame(summary_rows)
diag_df = diag_df.sort_values("elapsed_sec", ascending=False).reset_index(drop=True)

display(Markdown("### Diagnostic Summary"))
display(diag_df)

In [ ]:
# =============================================================================
# CELL 7: GUARDRAILS -- STATUS LABELS, STALENESS, COLOR-CODED SUMMARY
# =============================================================================

from datetime import datetime, timedelta

# ---------------------------------------------------------------------------
# Staleness thresholds (days). Tables with own date or freshness join are
# checked against these. Tables with no freshness mechanism are skipped.
# ---------------------------------------------------------------------------
STALENESS_THRESHOLDS = {
    "edwnpi.los_deal_current_fact":  3,
    "sandbox.rds_rec_model_originations": 3,
    "edwnpi.dealer_rollup_scd_current": 3,
    "edwnpi.crm_dealer_dim": 3,
    "edwnpi.dealer_attributes_pivot": 3,
    "edwnpi.date_dim": 30,

    "sandbox.student_loan_chime_flags": 7,
    "sandbox.temp_employment_type_ragu": 7,
    "sandbox.rds_blackbook_rollup": 7,
    "sandbox.kmx_approvals": 7,
    "sandbox.kmx_los_new_sp": 7,
    "sandbox.temp_prov_customer_credit_attributes_ragu": 7,
    "sandbox.temp_los_customer_credit_attributes_ragu": 7,
    "sandbox.loan_random_numbers": 7,
    "sandbox.temp_fraud_ragu": 7,
    "sandbox.temp_blackbook_values_ragu": 7,
    "sandbox.nonkmx_dealer_loss_data": 7,
}

# Tables that SHOULD have recent freshness data (flag if max_date is None)
EXPECTS_FRESHNESS = set(STALENESS_THRESHOLDS.keys())

today = pd.Timestamp.today().normalize()

def compute_status(row):
    if not row["reachable"]:
        return "DOWN"
    if row["total_rows"] == 0:
        return "EMPTY"
    if row["freshness_error"]:
        return "FRESHNESS_ERROR"

    table = row["table"]
    max_date_str = row["max_date"]

    if table in EXPECTS_FRESHNESS:
        if max_date_str in (None, "None", "NaT", "nan"):
            return "NO_FRESHNESS"
        try:
            max_dt = pd.Timestamp(max_date_str)
            days_stale = (today - max_dt).days
            threshold = STALENESS_THRESHOLDS.get(table, 7)
            if days_stale > threshold:
                return f"STALE ({days_stale}d)"
        except Exception:
            return "NO_FRESHNESS"

    return "OK"

guardrail_df = diag_df.copy()
guardrail_df["status"] = guardrail_df.apply(compute_status, axis=1)

# Reorder for readability
display_cols = ["status", "table", "used_by", "max_date", "recent_rows",
                "total_rows", "non_null_key_rows", "elapsed_sec",
                "error", "freshness_error"]
guardrail_df = guardrail_df[display_cols].sort_values(
    "status", key=lambda s: s.map(lambda v: 0 if v != "OK" else 1)
).reset_index(drop=True)

# ---------------------------------------------------------------------------
# Color-code by status
# ---------------------------------------------------------------------------
def highlight_row(row):
    status = row["status"]
    if status == "DOWN":
        return ["background-color: #d32f2f; color: white"] * len(row)
    elif status == "EMPTY":
        return ["background-color: #f57c00; color: white"] * len(row)
    elif status.startswith("STALE"):
        return ["background-color: #ffa726; color: black"] * len(row)
    elif status in ("NO_FRESHNESS", "FRESHNESS_ERROR"):
        return ["background-color: #ffee58; color: black"] * len(row)
    return [""] * len(row)

# ---------------------------------------------------------------------------
# Top-level verdict
# ---------------------------------------------------------------------------
statuses = set(guardrail_df["status"])
blockers = {s for s in statuses if s in ("DOWN", "EMPTY")}
warnings_set = {s for s in statuses if s.startswith("STALE") or s in ("NO_FRESHNESS", "FRESHNESS_ERROR")}

if blockers:
    verdict = "BLOCKED -- critical tables are down or empty. Do NOT run bareboned_ragu_new.ipynb."
    verdict_style = "color: #d32f2f; font-weight: bold; font-size: 16px"
elif warnings_set:
    verdict = "WARNINGS -- some tables are stale or missing freshness data. Review before running."
    verdict_style = "color: #f57c00; font-weight: bold; font-size: 16px"
else:
    verdict = "ALL CLEAR -- all tables are reachable and fresh."
    verdict_style = "color: #2e7d32; font-weight: bold; font-size: 16px"

display(Markdown(f"### Diagnostic Verdict"))
display(Markdown(f'<p style="{verdict_style}">{verdict}</p>'))

if blockers:
    blocked_tables = guardrail_df[guardrail_df["status"].isin(("DOWN", "EMPTY"))]["table"].tolist()
    display(Markdown("**Blocked by:** " + ", ".join(f"`{t}`" for t in blocked_tables)))

if warnings_set:
    warn_mask = guardrail_df["status"].apply(lambda s: s.startswith("STALE") or s in ("NO_FRESHNESS", "FRESHNESS_ERROR"))
    warn_tables = guardrail_df[warn_mask][["table", "status", "max_date"]].to_string(index=False)
    display(Markdown("**Warnings:**\n```\n" + warn_tables + "\n```"))

display(guardrail_df.style.apply(highlight_row, axis=1))
print("[PROGRESS] Guardrails Complete")

In [ ]:
# =============================================================================
# CELL 8: UPDATE CONFIGURATION (sandbox table refresh orchestration)
# =============================================================================
#
# Set update_tables = True to refresh the 6 user-owned sandbox tables after
# the diagnostic runs. DDL tables use a staging + atomic rename pattern so
# the public table name is always queryable, even mid-refresh.
#
# Execution order (heaviest DDL first so thread pool slots get claimed by the
# slowest jobs; the stored-procedure dispatch is last because it is cheap
# client-side and its runtime is dominated by server-side work).
# =============================================================================



TEMPTABLES_PATH = "ragu_temptables"

UPDATE_MAX_WORKERS = 5        # lower than probe (8) because DDL is heavy on shared edwnpi.los_deal_current_fact
UPDATE_STMT_TIMEOUT = 1800    # 30 minutes per statement

UPDATE_PLAN = [
    {"table": "sandbox.temp_fraud_ragu",                            "type": "ddl",       "key_col": "loan_id"},
    {"table": "sandbox.temp_blackbook_values_ragu",                 "type": "ddl",       "key_col": "account_number"},
    {"table": "sandbox.temp_los_customer_credit_attributes_ragu",   "type": "ddl",       "key_col": "customer_id"},
    {"table": "sandbox.temp_prov_customer_credit_attributes_ragu",  "type": "ddl",       "key_col": "customerid"},
    {"table": "sandbox.temp_employment_type_ragu",                  "type": "ddl",       "key_col": "account_number"},
    {"table": "sandbox.student_loan_chime_flags",                   "type": "procedure", "key_col": "loan_id",
     "call_sql": "CALL sandbox.student_loan_chime_flags();"},
]

print(f"update_tables = {update_tables}")
print(f"Tables in update plan: {len(UPDATE_PLAN)} "
      f"({sum(1 for e in UPDATE_PLAN if e['type']=='ddl')} DDL, "
      f"{sum(1 for e in UPDATE_PLAN if e['type']=='procedure')} procedure)")
print(f"Source file: {TEMPTABLES_PATH}")
print(f"Max parallel workers: {UPDATE_MAX_WORKERS}")

In [ ]:
# =============================================================================
# CELL 9: PARSER + STAGING/RENAME UPDATER + PARALLEL DISPATCH
# =============================================================================
#
# For each DDL entry, we:
#   1. Read the original CREATE body from ragu_temptables
#   2. Rewrite the `INTO <table>` clause to target `<table>_new` (staging)
#   3. Run a 10-step sequence per thread:
#        drop_staging -> build_new -> gate_count -> BEGIN -> drop_old ->
#        rename_curr_to_old -> rename_new_to_curr -> COMMIT -> grant -> drop_old_final
#   4. Each step is its own cur.execute() so failures attribute to the exact step.
#
# The stored-procedure entry (student_loan_chime_flags) bypasses all of this
# and runs its single CALL statement.
# =============================================================================

import re
from pathlib import Path


def _find_stmt_terminator(raw: str, start: int) -> int:
    """Scan forward from `start` and return the index of the next semicolon that
    terminates a SQL statement, respecting single-quoted strings, '' escapes,
    and -- line comments. Returns -1 if no terminator found."""
    i = start
    n = len(raw)
    in_string = False
    in_line_comment = False
    while i < n:
        c = raw[i]
        if in_line_comment:
            if c == "\n":
                in_line_comment = False
        elif in_string:
            if c == "'":
                if i + 1 < n and raw[i + 1] == "'":
                    i += 1  # skip escaped quote ''
                else:
                    in_string = False
        else:
            if c == "'":
                in_string = True
            elif c == "-" and i + 1 < n and raw[i + 1] == "-":
                in_line_comment = True
                i += 1
            elif c == ";":
                return i
        i += 1
    return -1


def extract_create_sql(raw: str, tbl: str) -> str:
    """Isolate the single SELECT INTO <tbl> statement body from ragu_temptables
    and rewrite its INTO target to <tbl>_new for the staging pattern.

    Boundaries:
      left:  end of `DROP TABLE IF EXISTS <tbl>;`
      right: the next statement-terminating ; after `INTO <tbl>` (respects
             single-quoted strings, '' escapes, and -- line comments)

    This avoids dependence on any particular verification-SELECT format
    (some tables use `select top 1 *`, others use custom diagnostic queries)."""
    drop_pat = re.compile(r"DROP\s+TABLE\s+IF\s+EXISTS\s+" + re.escape(tbl) + r"\s*;", re.IGNORECASE)
    drop_m = drop_pat.search(raw)
    if not drop_m:
        raise ValueError(f"Could not find DROP TABLE IF EXISTS {tbl}; in source file")

    into_pat = re.compile(r"INTO\s+" + re.escape(tbl) + r"\b", re.IGNORECASE)
    into_m = into_pat.search(raw, pos=drop_m.end())
    if not into_m:
        raise ValueError(f"Could not find 'INTO {tbl}' clause after DROP in source file")

    term_idx = _find_stmt_terminator(raw, into_m.end())
    if term_idx < 0:
        raise ValueError(f"Could not find statement-terminating ';' for SELECT INTO {tbl}")

    body = raw[drop_m.end(): term_idx + 1].strip()

    body_new = re.sub(
        r"(INTO\s+)" + re.escape(tbl) + r"\b",
        lambda m: m.group(1) + tbl + "_new",
        body,
        flags=re.IGNORECASE,
    )
    if body_new == body:
        raise ValueError(f"INTO {tbl} clause not found in CREATE body for staging rewrite")
    return body_new


def build_statements(entry: dict, raw_file: str) -> list:
    """Expand a single UPDATE_PLAN entry into an ordered list of statement dicts."""
    if entry["type"] == "procedure":
        return [{"step": "call_proc", "sql": entry["call_sql"]}]

    tbl = entry["table"]
    short = tbl.split(".")[-1]
    create_body = extract_create_sql(raw_file, tbl)

    # Atomic swap: `rename_curr_old` defers its commit so both renames land
    # in a single transaction committed at the end of `rename_new_curr`.
    # If `rename_new_curr` fails, the except block's conn.rollback() reverts
    # both renames together and the public name keeps its old data.
    return [
        {"step": "drop_staging",      "sql": f"DROP TABLE IF EXISTS {tbl}_new;"},
        {"step": "build_new",         "sql": create_body},
        {"step": "gate_count",        "sql": f"SELECT COUNT(*) FROM {tbl}_new;", "kind": "scalar"},
        {"step": "drop_old",          "sql": f"DROP TABLE IF EXISTS {tbl}_old;"},
        {"step": "rename_curr_old",   "sql": f"ALTER TABLE {tbl} RENAME TO {short}_old;",
                                      "skip_if_not_exists": tbl,
                                      "defer_commit": True},
        {"step": "rename_new_curr",   "sql": f"ALTER TABLE {tbl}_new RENAME TO {short};"},
        {"step": "grant",             "sql": f"CALL sandbox.util_table_grant('{short}');"},
        {"step": "drop_old_final",    "sql": f"DROP TABLE IF EXISTS {tbl}_old;"},
    ]


def update_table(entry: dict) -> dict:
    """Run one table's update sequence in its own connection. Returns a result dict."""
    t0 = time.time()
    result = {
        "table": entry["table"],
        "type": entry["type"],
        "ok": False,
        "failed_step": None,
        "steps_completed": [],
        "error": None,
        "gate_count": None,
        "elapsed_sec": None,
    }

    try:
        conn = pyodbc.connect(f"DSN={DSN}", timeout=15)
        conn.timeout = UPDATE_STMT_TIMEOUT
        conn.autocommit = False
    except Exception as e:
        result["error"] = f"Connection failed: {str(e)[:300]}"
        result["failed_step"] = "connect"
        result["elapsed_sec"] = round(time.time() - t0, 2)
        return result

    cur = conn.cursor()
    current_step = None
    try:
        for step in entry["statements"]:
            current_step = step["step"]

            if step.get("skip_if_not_exists"):
                schema, name = step["skip_if_not_exists"].split(".")
                cur.execute(
                    "SELECT 1 FROM pg_catalog.pg_tables WHERE schemaname = ? AND tablename = ?",
                    schema, name,
                )
                if cur.fetchone() is None:
                    result["steps_completed"].append(f"{current_step} (skipped, no prior table)")
                    continue

            if step.get("kind") == "scalar":
                cur.execute(step["sql"])
                val = cur.fetchone()[0]
                result["steps_completed"].append(f"{current_step}={val}")
                if current_step == "gate_count":
                    result["gate_count"] = int(val) if val is not None else 0
                    if result["gate_count"] == 0:
                        cur.execute(f"DROP TABLE IF EXISTS {entry['table']}_new;")
                        conn.commit()
                        raise RuntimeError("gate_count returned 0 rows; swap aborted, old table preserved")
            else:
                cur.execute(step["sql"])
                # Skip commit for steps flagged defer_commit; they stay in the
                # open transaction until a following step commits them atomically.
                if not step.get("defer_commit"):
                    conn.commit()
                result["steps_completed"].append(current_step)

        result["ok"] = True
    except Exception as e:
        result["error"] = f"{type(e).__name__}: {str(e)[:500]}"
        result["failed_step"] = current_step
        try:
            conn.rollback()
        except Exception:
            pass
    finally:
        try:
            cur.close()
            conn.close()
        except Exception:
            pass

    result["elapsed_sec"] = round(time.time() - t0, 2)
    return result


# ---------------------------------------------------------------------------
# Dispatch (only runs when update_tables is True)
# ---------------------------------------------------------------------------
if update_tables:
    raw_file = Path(TEMPTABLES_PATH).read_text(encoding="utf-8")

    for entry in UPDATE_PLAN:
        entry["statements"] = build_statements(entry, raw_file)

    total_steps = {e["table"]: len(e["statements"]) for e in UPDATE_PLAN}
    print(f"Prepared {len(UPDATE_PLAN)} tables for update. Per-table step counts: {total_steps}\n")

    print(f"Running updates in parallel (max_workers={UPDATE_MAX_WORKERS}) ...\n")
    upd_start = time.time()

    update_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=UPDATE_MAX_WORKERS) as executor:
        futures = {executor.submit(update_table, e): e["table"] for e in UPDATE_PLAN}
        for future in concurrent.futures.as_completed(futures):
            r = future.result()
            tag = "OK" if r["ok"] else f"FAIL@{r['failed_step']}"
            gate = f" gate={r['gate_count']}" if r.get("gate_count") is not None else ""
            print(f"  [{tag}] {r['table']}  ({r['elapsed_sec']}s){gate}")
            if not r["ok"]:
                print(f"           error: {r['error']}")
            update_results.append(r)

    upd_elapsed = round(time.time() - upd_start, 2)
    print(f"\nAll updates complete in {upd_elapsed}s")
else:
    print("update_tables = False  ->  skipping refresh. "
          "Set update_tables = True in Cell 8 and rerun from there to refresh the 6 sandbox tables.")
    update_results = []

In [ ]:
# =============================================================================
# CELL 10: POST-UPDATE INTEGRITY PROBE (moderate)
# =============================================================================
#
# Reuses the same check_table() probe from Cell 3 on just the 6 updated tables.
# Each probe runs in its own thread (one connection per table) so the whole
# integrity pass completes in ~one slowest-table's worth of wall clock time.
#
# Captures: row_count, non_null_key_rows, max_date, recent_rows, elapsed,
#           and any per-table freshness_error / probe error.
#
# Skipped entirely if update_tables = False.
# =============================================================================

if update_tables and update_results:
    updated_tables = {r["table"] for r in update_results}

    probe_cfgs = [cfg for cfg in TABLES_TO_CHECK if cfg["table"] in updated_tables]
    if len(probe_cfgs) != len(updated_tables):
        missing = updated_tables - {c["table"] for c in probe_cfgs}
        print(f"  WARNING: {len(missing)} updated tables have no matching TABLES_TO_CHECK config "
              f"and will be skipped in the integrity probe: {sorted(missing)}")

    print(f"Running post-update integrity probe on {len(probe_cfgs)} tables ...\n")
    iprobe_start = time.time()

    integrity_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(probe_cfgs) or 1) as executor:
        futures = {executor.submit(check_table, cfg): cfg["table"] for cfg in probe_cfgs}
        for future in concurrent.futures.as_completed(futures):
            r = future.result()
            tag = "OK" if r["reachable"] else "FAIL"
            print(f"  [{tag}] {r['table']}  ({r['elapsed_sec']}s)  "
                  f"rows={r['total_rows']}  max_date={r['max_date']}")
            integrity_results.append(r)

    iprobe_elapsed = round(time.time() - iprobe_start, 2)
    print(f"\nIntegrity probe complete in {iprobe_elapsed}s")

    upd_df = pd.DataFrame([
        {
            "table": r["table"],
            "type": r["type"],
            "update_ok": r["ok"],
            "failed_step": r["failed_step"],
            "gate_count": r["gate_count"],
            "update_elapsed_sec": r["elapsed_sec"],
            "update_error": r["error"],
        } for r in update_results
    ])

    int_df = pd.DataFrame([
        {
            "table": r["table"],
            "reachable_post": r["reachable"],
            "post_total_rows": r["total_rows"],
            "post_non_null_key_rows": r["non_null_key_rows"],
            "post_max_date": r["max_date"],
            "post_recent_rows": r["recent_rows"],
            "post_probe_error": r["error"],
            "post_freshness_error": r["freshness_error"],
        } for r in integrity_results
    ])

    update_summary_df = upd_df.merge(int_df, on="table", how="left")

    if "post_non_null_key_rows" in update_summary_df.columns and "post_total_rows" in update_summary_df.columns:
        update_summary_df["post_key_null_pct"] = (
            (update_summary_df["post_total_rows"] - update_summary_df["post_non_null_key_rows"])
            / update_summary_df["post_total_rows"].replace(0, pd.NA) * 100
        ).round(2)

    display_cols = ["table", "type", "update_ok", "failed_step", "gate_count",
                    "update_elapsed_sec", "post_total_rows", "post_key_null_pct",
                    "post_max_date", "post_recent_rows",
                    "update_error", "post_probe_error", "post_freshness_error"]
    existing_cols = [c for c in display_cols if c in update_summary_df.columns]
    update_summary_df = update_summary_df[existing_cols]

    display(Markdown("### Update + Integrity Summary"))
    display(update_summary_df)
else:
    update_summary_df = None
    print("Skipped -- update_tables = False or no update results to probe.")

In [ ]:
# =============================================================================
# CELL 11: UPDATE VERDICT (color-coded, step-level attribution)
# =============================================================================

if update_tables and update_summary_df is not None and not update_summary_df.empty:
    today_upd = pd.Timestamp.today().normalize()

    def _update_status(row):
        if not row.get("update_ok"):
            step = row.get("failed_step") or "unknown"
            if step in ("connect", "drop_staging", "build_new"):
                return "BUILD_FAILED"
            if step == "gate_count":
                return "GATE_FAILED"
            if step in ("drop_old", "rename_curr_old", "rename_new_curr"):
                return "SWAP_FAILED"
            if step == "grant":
                return "GRANT_FAILED"
            if step == "drop_old_final":
                return "CLEANUP_WARNING"
            if step == "call_proc":
                return "PROC_FAILED"
            return f"FAILED@{step}"

        if row.get("post_probe_error"):
            return "POST_PROBE_ERROR"
        if row.get("post_total_rows") in (None, 0) or pd.isna(row.get("post_total_rows")):
            return "UPDATE_SUCCESS_BUT_EMPTY"

        max_date_str = row.get("post_max_date")
        threshold = STALENESS_THRESHOLDS.get(row["table"], 7)
        if max_date_str not in (None, "None", "NaT", "nan") and not pd.isna(max_date_str):
            try:
                max_dt = pd.Timestamp(max_date_str)
                days_stale = (today_upd - max_dt).days
                if days_stale > threshold:
                    return f"UPDATE_SUCCESS_BUT_STALE ({days_stale}d)"
            except Exception:
                pass

        return "UPDATE_OK"

    verdict_df = update_summary_df.copy()
    verdict_df["update_status"] = verdict_df.apply(_update_status, axis=1)

    lead_cols = ["update_status", "table", "type", "failed_step", "gate_count",
                 "update_elapsed_sec", "post_total_rows", "post_key_null_pct",
                 "post_max_date"]
    tail_cols = [c for c in verdict_df.columns if c not in lead_cols + ["update_status"]]
    verdict_df = verdict_df[lead_cols + tail_cols]
    verdict_df = verdict_df.sort_values(
        "update_status", key=lambda s: s.map(lambda v: 0 if v != "UPDATE_OK" else 1)
    ).reset_index(drop=True)

    BLOCKERS = {"BUILD_FAILED", "GATE_FAILED", "SWAP_FAILED", "PROC_FAILED",
                "UPDATE_SUCCESS_BUT_EMPTY", "POST_PROBE_ERROR"}
    WARNINGS = {"GRANT_FAILED", "CLEANUP_WARNING"}

    def _row_style(row):
        s = row["update_status"]
        if s in BLOCKERS or s.startswith("FAILED@"):
            return ["background-color: #d32f2f; color: white"] * len(row)
        if s in WARNINGS:
            return ["background-color: #f57c00; color: white"] * len(row)
        if s.startswith("UPDATE_SUCCESS_BUT_STALE"):
            return ["background-color: #ffa726; color: black"] * len(row)
        return [""] * len(row)

    statuses = set(verdict_df["update_status"])
    blocker_hits = {s for s in statuses if s in BLOCKERS or s.startswith("FAILED@")}
    warning_hits = {s for s in statuses if s in WARNINGS or s.startswith("UPDATE_SUCCESS_BUT_STALE")}

    if blocker_hits:
        verdict_msg = "UPDATE BLOCKED -- one or more tables failed or produced empty/unhealthy output. Review before running bareboned_ragu_new.ipynb."
        verdict_color = "#d32f2f"
    elif warning_hits:
        verdict_msg = "UPDATE WARNINGS -- tables refreshed but grants, cleanup, or freshness have issues. Review before relying on them."
        verdict_color = "#f57c00"
    else:
        verdict_msg = "ALL UPDATES CLEAR -- all 6 tables refreshed and verified."
        verdict_color = "#2e7d32"

    display(Markdown("### Update Verdict"))
    display(Markdown(
        f'<p style="color: {verdict_color}; font-weight: bold; font-size: 16px">{verdict_msg}</p>'
    ))

    if blocker_hits:
        blocked = verdict_df[verdict_df["update_status"].apply(
            lambda s: s in BLOCKERS or s.startswith("FAILED@"))]["table"].tolist()
        display(Markdown("**Blocked / failed:** " + ", ".join(f"`{t}`" for t in blocked)))

    if warning_hits:
        warned = verdict_df[verdict_df["update_status"].apply(
            lambda s: s in WARNINGS or s.startswith("UPDATE_SUCCESS_BUT_STALE"))][
            ["table", "update_status", "post_max_date"]].to_string(index=False)
        display(Markdown("**Warnings:**\n```\n" + warned + "\n```"))

    display(verdict_df.style.apply(_row_style, axis=1))
else:
    print("Skipped -- update_tables = False or nothing to summarize.")
print("[PROGRESS] Diagnostic Complete")